# Add & Norm（残差接続と層正規化）

このノートブックでは、Multi-Head Attention の出力を
次の層へ渡す前に行う **Add & Norm** の処理を学びます。

書籍 4-11 節（図4.37〜4.39、式4-6〜4-8）の内容をカバーします。

## 目次
1. Add & Norm の全体像（図4.37）
2. 4つのステップの概要
3. ステップ1: ヘッドの結合（Concat）
4. ステップ2: 線形変換（Linear）— W_o を掛ける
5. ステップ3: 残差接続（Add / Skip Connection）
6. ステップ4: 層正規化（Layer Normalization）— 式(4-7), (4-8)
7. Add & Norm を一気通貫で実行する
8. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 日本語フォント設定（macOS）
plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False

# 書籍と同じ設定
n_tokens = 7       # トークン数（"Mount Fuji looks beautiful in spring ."）
d_model = 6         # 埋め込み次元
n_heads = 3         # ヘッド数
d_k = d_model // n_heads  # 各ヘッドの次元 = 2

words = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

print(f"トークン数: {n_tokens}")
print(f"埋め込み次元 d_model: {d_model}")
print(f"ヘッド数: {n_heads}")
print(f"各ヘッドの次元 d_k: {d_k}")

## 1. Add & Norm の全体像（図4.37）

Multi-Head Attention の出力は、そのまま次の処理に渡すのではなく、
**Add & Norm** という後処理を経ます。

```
入力 X (7×6) ──────────────────────────┐
    │                                   │（Skip Connection）
    ↓                                   │
Multi-Head Attention                    │
    ↓                                   │
① Concat (各ヘッドの出力を結合)          │
    ↓                                   │
② Linear (W_o を掛ける)                 │
    ↓                                   │
③ Add ←─────────────────────────────────┘
    ↓
④ Norm (Layer Normalization)
    ↓
出力 (7×6) → 次の処理（Feed Forward）へ
```

### なぜ Add & Norm が必要なのか？

| 処理 | 役割 |
|------|------|
| **Add（残差接続）** | 元の入力の情報を保持する。深い層でも勾配が消えにくくなる |
| **Norm（正規化）** | 値のスケールを安定させ、学習を安定化させる |

## 2. 4つのステップの概要

Add & Norm は以下の **4つのステップ** で構成されています（図4.38）。

| ステップ | 処理名 | 入力 → 出力 | 内容 |
|:--------:|--------|-------------|------|
| ① | **Concat** | 3つの (7×2) → (7×6) | 各ヘッドの出力を横に結合 |
| ② | **Linear** | (7×6) → (7×6) | 重み行列 W_o (6×6) を掛ける |
| ③ | **Add** | (7×6) + (7×6) → (7×6) | 元の入力 X を足す（Skip Connection）|
| ④ | **Norm** | (7×6) → (7×6) | Layer Normalization で正規化 |

すべてのステップで **形状は 7×6 のまま変わらない** のがポイントです。

## 3. ステップ1: ヘッドの結合（Concat）

前回のノートブックで学んだ通り、3つのヘッドの出力を横に連結します。

```
Head 1 (7×2) | Head 2 (7×2) | Head 3 (7×2)  →  Concat  →  (7×6)
```

まず、ダミーデータを使って各ヘッドの出力を作成しましょう。

In [ ]:
# まず入力 X を作成（前回のノートブックの続き）
np.random.seed(42)
X = np.round(np.random.randn(n_tokens, d_model) * 0.5 + 0.5, 2)

print("入力 X (7×6):")
print(X)
print(f"形状: {X.shape}")

# Softmax 関数の定義
def softmax(x):
    """各行に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# 3つのヘッドを計算
head_outputs = []
for h in range(n_heads):
    np.random.seed(h * 100)  # 各ヘッドで異なる重み
    W_Q = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
    W_K = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
    W_V = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
    
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    
    attn = softmax(Q @ K.T / np.sqrt(d_k))
    output = attn @ V  # (7×2)
    head_outputs.append(output)
    print(f"\nHead {h+1} の出力 (7×2):")
    print(np.round(output, 4))

# ① Concat: 3つのヘッドを横に結合
concat_output = np.concatenate(head_outputs, axis=1)  # (7×6)
print(f"\n=== ① Concat の結果 ===")
print(f"形状: {concat_output.shape}  (7×6 に戻った！)")
print(np.round(concat_output, 4))

## 4. ステップ2: 線形変換（Linear）— W_o を掛ける（図4.38）

Concat の結果に **重み行列 $W_o$**（6×6）を掛けます。

$$\text{Linear}(\text{Concat}) = \text{Concat} \times W_o$$

$$\underbrace{(7 \times 6)}_{\text{Concat}} \times \underbrace{(6 \times 6)}_{W_o} = \underbrace{(7 \times 6)}_{\text{出力}}$$

### なぜ線形変換が必要なのか？

| 理由 | 説明 |
|------|------|
| **ヘッド間の情報統合** | 各ヘッドが捉えた特徴を混ぜ合わせる |
| **次元の調整** | Concat で単純に結合しただけの表現を、意味のある表現に変換 |
| **学習可能な変換** | $W_o$ は学習で最適化されるため、タスクに適した変換が学ばれる |

$W_o$ は学習によって値が決まる **パラメータ** です。

In [ ]:
# ② Linear: W_o を掛ける
np.random.seed(999)
W_o = np.round(np.random.randn(d_model, d_model) * 0.3, 3)  # (6×6)

print("=== ② Linear（線形変換）===")
print(f"Concat の形状: {concat_output.shape}  (7×6)")
print(f"W_o の形状:    {W_o.shape}  (6×6)")

linear_output = concat_output @ W_o  # (7×6) × (6×6) = (7×6)
print(f"出力の形状:    {linear_output.shape}  (7×6)")
print()
print("W_o (6×6):")
print(W_o)
print()
print("Linear の出力 (7×6):")
print(np.round(linear_output, 4))

## 5. ステップ3: 残差接続（Add / Skip Connection）（図4.39）

ここが Add & Norm の **Add** の部分です。

線形変換の出力に、**元の入力 X をそのまま足します**。

$$\text{Add} = \text{Linear}(\text{Concat}) + X$$

### なぜ元の入力を足すのか？（Skip Connection）

これを **残差接続（Residual Connection）** または **スキップ接続（Skip Connection）** と呼びます。

```
入力 X ──────────────┐
    │                │
    ↓                │
  何らかの変換 f(X)    │
    │                │
    ↓                │
  f(X) + X ←─────────┘  ← ここが Add（残差接続）
```

| メリット | 説明 |
|----------|------|
| **勾配の消失を防ぐ** | 深い層（N=6回繰り返し）でも、元の入力の情報が直接伝わる |
| **学習の安定化** | 変換 f(X) がうまく学習できなくても、最低限 X はそのまま通る |
| **「差分」を学習** | f(X) は「X からの変化量」だけ学習すればよい → 学習が楽になる |

ResNet（画像認識の有名モデル）でも同じアイデアが使われています。

In [ ]:
# ③ Add: 残差接続（Skip Connection）

print("=== ③ Add（残差接続）===")
print(f"Linear出力の形状: {linear_output.shape}  (7×6)")
print(f"元の入力 X の形状: {X.shape}  (7×6)")

# 単純に足すだけ！
add_output = linear_output + X  # (7×6) + (7×6) = (7×6)
print(f"Add の出力の形状:  {add_output.shape}  (7×6)")
print()

# 具体的な計算を1つのトークンで見てみる
token_idx = 0
print(f'--- "{words[token_idx]}" の計算例 ---')
print(f"Linear出力: {np.round(linear_output[token_idx], 4)}")
print(f"元の入力 X: {np.round(X[token_idx], 4)}")
print(f"Add の結果: {np.round(add_output[token_idx], 4)}")
print()
print("ポイント: Linear の出力 + 元の入力 X を要素ごとに足すだけ！")
print("→ 元の入力の情報が確実に残る（学習が深くても情報が消えにくい）")

In [ ]:
# Skip Connection の効果を可視化

fig, axes = plt.subplots(1, 4, figsize=(18, 5),
                         gridspec_kw={'width_ratios': [1, 1, 0.3, 1]})

# 元の入力 X
ax = axes[0]
im = ax.imshow(X, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=2)
ax.set_title('元の入力 X\n(7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xlabel('次元', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

# Linear の出力
ax = axes[1]
im = ax.imshow(linear_output, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=2)
ax.set_title('Linear(Concat)\n(7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xlabel('次元', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

# + 記号
ax = axes[2]
ax.text(0.5, 0.5, '+', fontsize=40, ha='center', va='center', fontweight='bold')
ax.axis('off')

# Add の結果
ax = axes[3]
im = ax.imshow(add_output, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=2)
ax.set_title('Add の結果\n= Linear + X (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
ax.set_xlabel('次元', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('図4.39: Skip Connection（元の入力 X を足す）', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. ステップ4: 層正規化（Layer Normalization）— 式(4-7), (4-8)

Add の結果を **正規化** して、値のスケールを安定させます。

### 正規化とは？（式4-6）

一般的な正規化は「値を 0〜1 の範囲に収める」処理です：

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \quad \text{（式4-6: 一般的な正規化）}$$

しかし、Transformer で使うのはこれではなく **Layer Normalization** です。

### Layer Normalization（式4-7, 4-8）

各トークン（各行）ごとに、平均と標準偏差を使って正規化します。

$$x'_i = a_i \cdot \frac{x_i - \mu_x}{\sigma_x + \varepsilon} + b_i \quad \text{（式4-7）}$$

ここで：

$$\mu_x = \frac{1}{d} \sum_{k=1}^{d} x_k \quad \text{（平均）}$$

$$\sigma_x = \sqrt{\frac{1}{d} \sum_{k=1}^{d} (x_k - \mu_x)^2} \quad \text{（標準偏差）}$$

$\text{（式4-8）}$

### 各記号の意味

| 記号 | 意味 | 備考 |
|------|------|------|
| $x_i$ | 正規化前の値（各次元の値）| |
| $\mu_x$ | その行（トークン）の平均 | 6次元の平均 |
| $\sigma_x$ | その行（トークン）の標準偏差 | 6次元の標準偏差 |
| $\varepsilon$ | 分母が0になるのを防ぐ小さな値 | $1.0 \times 10^{-6}$ |
| $a_i$ | スケーリングパラメータ | **学習で決まる** |
| $b_i$ | シフトパラメータ | **学習で決まる** |

### ポイント

- $a_i$ と $b_i$ は **学習可能なパラメータ** です（固定値ではない）
- $(x_i - \mu_x) / \sigma_x$ で平均0・分散1に正規化した後、$a_i$ と $b_i$ で再調整する
- **行ごと（トークンごと）** に正規化する → 各トークンの特徴量が安定する

In [ ]:
# Layer Normalization を手動で実装する

epsilon = 1e-6  # ε = 1.0 × 10⁻⁶

# 学習可能パラメータ（初期値はスケール=1, シフト=0 が一般的）
a = np.ones(d_model)   # a_i: スケーリングパラメータ (6次元)
b = np.zeros(d_model)  # b_i: シフトパラメータ (6次元)

print("=== Layer Normalization の計算過程 ===")
print(f"ε (epsilon) = {epsilon}")
print(f"a (スケーリング) = {a}  ← 学習で変わる（初期値=1）")
print(f"b (シフト) = {b}  ← 学習で変わる（初期値=0）")
print()

# 1つのトークンで詳しく計算を見る
token_idx = 0
x = add_output[token_idx]  # "Mount" の6次元ベクトル
print(f'--- "{words[token_idx]}" の計算例 ---')
print(f"入力 x = {np.round(x, 4)}")
print()

# 式(4-8): 平均を計算
mu = np.mean(x)
print(f"式(4-8) 平均 μ_x = (1/{d_model}) × Σx_k = {mu:.4f}")

# 式(4-8): 標準偏差を計算
sigma = np.std(x)  # np.std は母標準偏差（1/N）を計算
print(f"式(4-8) 標準偏差 σ_x = √((1/{d_model}) × Σ(x_k - μ_x)²) = {sigma:.4f}")
print()

# 式(4-7): 正規化
x_normalized = (x - mu) / (sigma + epsilon)
print(f"正規化: (x - μ) / (σ + ε) = {np.round(x_normalized, 4)}")
print(f"  → 平均: {np.mean(x_normalized):.6f}  （ほぼ0）")
print(f"  → 標準偏差: {np.std(x_normalized):.6f}  （ほぼ1）")
print()

# a_i と b_i を適用
x_layernorm = a * x_normalized + b
print(f"式(4-7): x' = a × (x-μ)/(σ+ε) + b = {np.round(x_layernorm, 4)}")
print(f"  （a=1, b=0 なので正規化結果と同じ。学習が進むと値が変わる）")

In [ ]:
# 全トークンに Layer Normalization を適用

def layer_norm(x, a, b, epsilon=1e-6):
    """Layer Normalization の実装
    
    式(4-7): x'_i = a_i * (x_i - μ_x) / (σ_x + ε) + b_i
    式(4-8): μ_x = (1/d) * Σx_k,  σ_x = √((1/d) * Σ(x_k - μ_x)²)
    
    Parameters:
        x: 入力行列 (n_tokens × d_model)
        a: スケーリングパラメータ (d_model,)
        b: シフトパラメータ (d_model,)
        epsilon: ゼロ除算防止の小さな値
    """
    # 各行（トークン）ごとに平均と標準偏差を計算
    mu = np.mean(x, axis=1, keepdims=True)      # (7, 1)
    sigma = np.std(x, axis=1, keepdims=True)     # (7, 1)
    
    # 正規化 → スケーリング → シフト
    x_norm = a * (x - mu) / (sigma + epsilon) + b
    return x_norm, mu, sigma

# ④ Norm: Layer Normalization を適用
norm_output, mu_all, sigma_all = layer_norm(add_output, a, b, epsilon)

print("=== ④ Norm（Layer Normalization）===")
print(f"入力の形状:  {add_output.shape}  (7×6)")
print(f"出力の形状:  {norm_output.shape}  (7×6)")
print()

# 各トークンの平均と標準偏差を表示
print("各トークンの正規化前後の統計:")
print(f"{'トークン':12s} | {'正規化前の平均':>14s} | {'正規化前の標準偏差':>18s} | {'正規化後の平均':>14s} | {'正規化後の標準偏差':>18s}")
print("-" * 90)
for i, word in enumerate(words):
    before_mean = np.mean(add_output[i])
    before_std = np.std(add_output[i])
    after_mean = np.mean(norm_output[i])
    after_std = np.std(norm_output[i])
    print(f"{word:12s} | {before_mean:14.4f} | {before_std:18.4f} | {after_mean:14.6f} | {after_std:18.6f}")

print()
print("ポイント: 正規化後は各トークンの平均≈0、標準偏差≈1 になる")

In [ ]:
# Layer Normalization の効果を可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 正規化前
ax = axes[0]
im = ax.imshow(add_output, cmap='RdBu_r', aspect='auto')
ax.set_title('正規化前（Add の出力）', fontsize=13, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=10)
ax.set_xlabel('次元', fontsize=11)
for i in range(n_tokens):
    for j in range(d_model):
        ax.text(j, i, f'{add_output[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

# 正規化後
ax = axes[1]
im = ax.imshow(norm_output, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax.set_title('正規化後（Layer Norm の出力）', fontsize=13, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=10)
ax.set_xlabel('次元', fontsize=11)
for i in range(n_tokens):
    for j in range(d_model):
        ax.text(j, i, f'{norm_output[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Layer Normalization の効果：各行の平均→0、標準偏差→1', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Layer Normalization のイメージ: 各トークンの値の分布を比較

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(n_tokens, 4)):  # 最初の4トークン
    # 正規化前
    ax = axes[0, i]
    ax.bar(range(d_model), add_output[i], color='steelblue', alpha=0.8)
    ax.axhline(y=np.mean(add_output[i]), color='red', linestyle='--', label=f'平均={np.mean(add_output[i]):.2f}')
    ax.set_title(f'"{words[i]}" 正規化前', fontsize=10, fontweight='bold')
    ax.set_xlabel('次元', fontsize=9)
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=8)
    
    # 正規化後
    ax = axes[1, i]
    ax.bar(range(d_model), norm_output[i], color='coral', alpha=0.8)
    ax.axhline(y=0, color='red', linestyle='--', label='平均=0')
    ax.set_title(f'"{words[i]}" 正規化後', fontsize=10, fontweight='bold')
    ax.set_xlabel('次元', fontsize=9)
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=8)

plt.suptitle('Layer Normalization: 各トークンの値の分布を安定させる', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("上段: 正規化前 → 各トークンで値のスケールがバラバラ")
print("下段: 正規化後 → 平均0・標準偏差1 に統一され、学習が安定する")

## 7. Add & Norm を一気通貫で実行する

4つのステップをまとめて実行し、全体の流れを確認しましょう。

In [ ]:
# Add & Norm の全体の流れをまとめて実行

print("=" * 60)
print("  Add & Norm の全体の流れ")
print("=" * 60)
print()

# 入力
print(f"【入力】X の形状: {X.shape}  (7×6)")
print()

# Multi-Head Attention（省略：前回のノートブックで詳しく学んだ）
print("--- Multi-Head Attention（前回のノートブック参照）---")
for h in range(n_heads):
    print(f"  Head {h+1} の出力: {head_outputs[h].shape}  (7×2)")
print()

# ① Concat
print(f"① Concat: 3つの(7×2)を横に結合 → {concat_output.shape}  (7×6)")

# ② Linear
print(f"② Linear: Concat × W_o(6×6) → {linear_output.shape}  (7×6)")

# ③ Add
print(f"③ Add: Linear + X → {add_output.shape}  (7×6)  ← Skip Connection")

# ④ Norm
print(f"④ Norm: LayerNorm(Add) → {norm_output.shape}  (7×6)")

print()
print(f"【出力】形状: {norm_output.shape}  (7×6)")
print()
print("重要: 入力も出力も同じ 7×6 の形状！")
print("→ だから N=6 回繰り返すことができる")

In [ ]:
# Add & Norm の4ステップを可視化

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 共通の色範囲
vmin, vmax = -2, 2

# (0,0) 入力 X
ax = axes[0, 0]
im = ax.imshow(X, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_title('入力 X (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# (0,1) ① Concat
ax = axes[0, 1]
im = ax.imshow(concat_output, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_title('① Concat (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# (0,2) ② Linear
ax = axes[0, 2]
im = ax.imshow(linear_output, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_title('② Linear (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# (1,0) ③ Add
ax = axes[1, 0]
im = ax.imshow(add_output, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_title('③ Add = Linear + X (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# (1,1) ④ Norm
ax = axes[1, 1]
im = ax.imshow(norm_output, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_title('④ Norm (7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# (1,2) フロー図
ax = axes[1, 2]
ax.axis('off')
flow_text = (
    "Add & Norm の流れ\n"
    "─────────────\n\n"
    "入力 X (7×6)\n"
    "    ↓ MHA\n"
    "① Concat (7×6)\n"
    "    ↓ × W_o\n"
    "② Linear (7×6)\n"
    "    ↓ + X\n"
    "③ Add (7×6)\n"
    "    ↓ LN\n"
    "④ Norm (7×6)\n\n"
    "形状はずっと 7×6！"
)
ax.text(0.5, 0.5, flow_text, fontsize=13, ha='center', va='center',
        fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Add & Norm の4ステップ（図4.38）', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. まとめ

| ポイント | 内容 |
|----------|------|
| **Add & Norm** | Multi-Head Attention の後処理。4ステップで構成 |
| **① Concat** | 各ヘッドの出力(7×2)を横に結合 → (7×6) |
| **② Linear** | 重み行列 $W_o$ (6×6) を掛けてヘッド間の情報を統合 |
| **③ Add（Skip Connection）** | 元の入力 X を足す。勾配消失を防ぎ、学習を安定化 |
| **④ Norm（Layer Norm）** | 各行ごとに平均0・標準偏差1に正規化 |
| **式(4-7)** | $x'_i = a_i \cdot (x_i - \mu_x) / (\sigma_x + \varepsilon) + b_i$ |
| **式(4-8)** | $\mu_x = (1/d)\sum x_k$, $\sigma_x = \sqrt{(1/d)\sum(x_k-\mu_x)^2}$ |
| **$a_i, b_i$** | 学習可能なパラメータ（正規化後の微調整用）|
| **$\varepsilon$** | $1.0 \times 10^{-6}$（ゼロ除算防止）|
| **形状の保存** | 入力も出力も 7×6 → N回繰り返し可能 |

### エンコーダ1層の全体像

```
入力 X (7×6)
  ├── Positional Encoding を加算
  ↓
  ├──────────────────────────────── Skip Connection ──┐
  ↓                                                  │
  Multi-Head Attention                               │
  ↓                                                  │
  Concat → Linear                                    │
  ↓                                                  │
  Add ←──────────────────────────────────────────────┘
  ↓
  Layer Norm
  ↓
  出力 (7×6) → Feed Forward Network へ
```

## 次のステップ

次は **Feed Forward Network** を学びます。
Add & Norm の出力を受け取り、さらに変換を行います。
（その後にもう一度 Add & Norm を通ります）